# 02 — Split 5 task và nghiệm thu dữ liệu thật

**Ngày tạo:** 07/10/2026. **Protocol đã chốt:** 100 + 25 + 25 + 25 + 25 SKU, seed mapping 0, phân tầng theo 17 nhóm hàng. Tái sử dụng config 224 slot/225 outputs hiện có; 24 slot dự phòng không phải task dữ liệu thứ 6.

**Mục đích:** tạo bộ train/validation/test độc lập theo nhóm ảnh và COCO JSON cho từng task. Đợt này dùng **ảnh checkout thật**; ảnh ghép chưa đưa vào train do mask/bbox/provenance còn chưa nghiệm thu. Notebook review nhãn riêng sẽ xử lý nguồn synth.

**Split policy:** giữ mọi nhóm chứa val2019 trong train vì background/statistics synth được lấy từ nguồn này; lấy validation/test từ các nhóm chỉ có test2019. Phần test2019 còn lại được dùng train. Đây là split nghiên cứu độc lập trên RPC, **không phải benchmark official test2019 nguyên vẹn**. Mục tiêu test ~6.000, val ~1.500, pilot ~3.000 ảnh; group được giữ nguyên nên có thể vượt vài ảnh. Dùng công cụ chia tập hiện có với stratify=none vì nhóm RPC trộn các mức độ đông. Min coverage: test 50 và val 15 đối tượng/SKU.

**Input bất biến:** data/archive, configs/tasks_100-4x25_seed0.json. **Output:** data/processed/rpc_100-4x25_seed0_native_v1 và config lock configs/data/. Giữ pixel/bbox kích thước gốc, không resize/copy ảnh ở bước split; processor 640/800 nghiệm thu sau. Mọi cell acceptance phải đạt trước khi ghi release manifest.


In [1]:
import os,sys,json,hashlib,uuid,subprocess,platform
from importlib.metadata import version,PackageNotFoundError
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
REPO=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pdp/main.py').exists())
if str(REPO) not in sys.path:sys.path.insert(0,str(REPO))
from autocheckout.io import load_json,save_json,md5_file
from autocheckout.taskcfg import TaskConfig
from autocheckout.groups import parse_file_name,sku_multiset
from tools.resize import merge_annotations
from tools.make_split import main as make_split
from tools.make_task_json import main as make_task_json
from tools.make_task_config import build_task_config
TZ=ZoneInfo('Asia/Bangkok')
def now():return datetime.now(TZ).isoformat(timespec='seconds')
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''):h.update(chunk)
    return h.hexdigest()
RAW=Path(os.environ.get('AUTOCHECKOUT_DATA_ROOT',str(REPO/'data/archive'))).resolve()
OUT=Path(os.environ.get('AUTOCHECKOUT_SPLIT_OUTPUT',str(REPO/'data/processed/rpc_100-4x25_seed0_native_v1'))).resolve()
assert RAW!=OUT and not OUT.is_relative_to(RAW),'Output phải tách khỏi archive gốc'
CFG=REPO/'configs/tasks_100-4x25_seed0.json'
PARAMS={'seed':0,'max_tries':20,'test_per_level':2000,'val_per_level':500,'pilot':3000,
        'min_test_objects':50,'min_val_objects':15,'stratify':'none','cap':6000}
run={'run_id':str(uuid.uuid4()),'purpose':'5-task real-checkout split and acceptance','started_at':now(),
     'finished_at':None,'timezone':'Asia/Bangkok','python':sys.version.split()[0],
     'git_commit':subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip(),
     'raw_root':str(RAW),'output_root':str(OUT),'params':PARAMS,'status':'RUNNING'}
ARTIFACT=REPO/'runs/data_preprocessing/split'/run['run_id'];ARTIFACT.mkdir(parents=True,exist_ok=True)
source_hashes={name:sha256(RAW/name) for name in ['instances_val2019.json','instances_test2019.json']}
config_hash=sha256(CFG)
release_input={'source_sha256':source_hashes,'task_config_sha256':config_hash,'params':PARAMS,
               'train_source':'real','coordinates':'native_pixels'}
if (OUT/'release_manifest.json').exists():
    assert load_json(OUT/'release_manifest.json')['input']==release_input,'Input/policy changed: create a new release output, do not overwrite lock'
OUT.mkdir(parents=True,exist_ok=True)
checks={}
def dependency_version(name):
    try:return version(name)
    except PackageNotFoundError:return None
run.update({'git_commit':subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip(),
            'git_dirty':bool(subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip()),
            'host':platform.node(),'platform':platform.platform(),
            'dependency_versions':{name:dependency_version(name) for name in ['numpy','Pillow','tqdm']},
            'gpu_used':False})
print(json.dumps(run,ensure_ascii=False,indent=2))


{
  "run_id": "6b918406-79ea-4531-b3f0-76af0d6d265b",
  "purpose": "5-task real-checkout split and acceptance",
  "started_at": "2026-10-07T16:02:43+07:00",
  "finished_at": null,
  "timezone": "Asia/Bangkok",
  "python": "3.14.4",
  "git_commit": "6269c0e083f092995e608ec3feed337c0e50b8a5",
  "raw_root": "/home/sakana/Code/AutoCheckout-CL/data/archive",
  "output_root": "/home/sakana/Code/AutoCheckout-CL/data/processed/rpc_100-4x25_seed0_native_v1",
  "params": {
    "seed": 0,
    "max_tries": 20,
    "test_per_level": 2000,
    "val_per_level": 500,
    "pilot": 3000,
    "min_test_objects": 50,
    "min_val_objects": 15,
    "stratify": "none",
    "cap": 6000
  },
  "status": "RUNNING",
  "git_dirty": true,
  "host": "Laptop",
  "platform": "Linux-7.0.0-30-generic-x86_64-with-glibc2.43",
  "dependency_versions": {
    "numpy": "2.3.5",
    "Pillow": "12.1.1",
    "tqdm": "4.67.3"
  },
  "gpu_used": false
}


## 1. Nghiệm thu catalog và mapping đã chốt

Đối chiếu 200 SKU của val/test với config 5 task; giữ mapping hiện có và số slot để model không đổi kiến trúc classifier âm thầm. Ghi snapshot config vào release. Không đưa task dự phòng vào training.


In [2]:
cocos={s:load_json(RAW/f'instances_{s}.json') for s in ['val2019','test2019']}
config=TaskConfig.load(CFG)
assert [t.num_classes for t in config.data_tasks]==[100,25,25,25,25]
assert set(config.rpc_to_label())==set(range(1,201))
assert set(config.rpc_to_label().values())==set(range(200))
assert [config.seen_classes(t) for t in range(1,6)]==[100,125,150,175,200]
regenerated=build_task_config(cocos['val2019']['categories'],[100,25,25,25,25],reserved=24,seed=0,name=config.name)
assert regenerated.to_dict()==config.to_dict(),'Existing mapping differs from source catalog/seed'
for s,d in cocos.items():
    assert {c['id'] for c in d['categories']}==set(range(1,201))
    assert len({i['id'] for i in d['images']})==len(d['images'])
    assert len({a['id'] for a in d['annotations']})==len(d['annotations'])
    assert {a['image_id'] for a in d['annotations']}<={i['id'] for i in d['images']}
save_json(OUT/'task_config.json',config.to_dict(),indent=1)
checks['protocol_catalog_mapping']='PASS'
print('PASS: 5 data tasks, 200 SKU, 17 groups/task, mapping and source catalog match; reserved slots stay unused')


PASS: 5 data tasks, 200 SKU, 17 groups/task, mapping and source catalog match; reserved slots stay unused


## 2. Gộp annotation theo pixel gốc

Gán image/annotation IDs mới không đụng nhau; lưu source và orig_id. file_name là val2019/<name> hoặc test2019/<name> để loader dùng archive làm image root. Không làm tròn/scale bbox hoặc area ở chế độ native.


In [3]:
merged=merge_annotations(cocos,size=None)
assert len(merged['images'])==sum(len(d['images']) for d in cocos.values())==30000
assert len(merged['annotations'])==sum(len(d['annotations']) for d in cocos.values())
assert len({i['id'] for i in merged['images']})==len(merged['images'])
assert len({a['id'] for a in merged['annotations']})==len(merged['annotations'])
original_ann={(s,a['id']):a for s,d in cocos.items() for a in d['annotations']}
for a in merged['annotations']:
    old=original_ann[a['source'],a['orig_id']]
    assert a['bbox']==old['bbox'] and a['area']==old['area'] and a['category_id']==old['category_id']
for image in merged['images']:
    assert (RAW/image['file_name']).is_file(),image['file_name']
ANN=OUT/'ann/checkout_native.json';save_json(ANN,merged)
checks['native_merge_no_lost_objects']='PASS'
print('PASS:',len(merged['images']),'images;',len(merged['annotations']),'objects; native geometry unchanged')
del original_ann,cocos


PASS: 30000 images; 367935 objects; native geometry unchanged


## 3. Tạo split theo group bằng công cụ hiện có

Group = quan hệ cùng suffix ảnh hoặc cùng SKU multiset, lấy connected components. Seed thử chỉ theo min coverage đã định trước, không theo kết quả model. Không sửa configs/splits cũ; manifest split mới thuộc release này.


In [4]:
make_split(['--ann',str(ANN),'--out-dir',str(OUT/'splits'),'--config-dir',str(OUT/'split_config'),
            *[a for k,v in PARAMS.items() if k!='cap' for a in ('--'+k.replace('_','-'),str(v))]])
split_config_paths=list((OUT/'split_config').glob('rpc_checkout_seed*.json'))
assert len(split_config_paths)==1
split_cfg=load_json(split_config_paths[0])
splits={name:load_json(OUT/f'splits/{name}.json') for name in ['train','val','test','train_pilot']}
print('Split seed accepted:',split_cfg['seed'],'| SKU mapping seed:',config.seed)


train        groups   2504  images  22494  objects  276091  per level {'easy': 7502, 'medium': 7527, 'hard': 7465}
val          groups    167  images   1503  objects   18378  per level {'easy': 490, 'medium': 504, 'hard': 509}
test         groups    663  images   6003  objects   73466  per level {'easy': 2008, 'medium': 1969, 'hard': 2026}
train_pilot  groups    333  images   3002  objects   36987  per level {'easy': 982, 'medium': 1024, 'hard': 996}
seeds tried: [0]; accepted seed 0; 3323 groups span levels; wrote /home/sakana/Code/AutoCheckout-CL/data/processed/rpc_100-4x25_seed0_native_v1/split_config/rpc_checkout_seed0.json
Split seed accepted: 0 | SKU mapping seed: 0


## 4. Nghiệm thu partition, coverage, leakage và nguồn background

Phải giữ group/suffix/SKU multiset nguyên vẹn giữa train/val/test, pilot là subset train, full partition không mất ảnh/nhãn. Tính SHA-256 mới trên 30.000 file ảnh để kiểm tra exact duplicates qua split. Đây chưa thay thế tất cả perceptual near-duplicate detection. Tên nguồn của 208 nền phải thuộc train, không thuộc holdout.


In [5]:
main_names=['train','val','test']
ids={name:{i['id'] for i in splits[name]['images']} for name in splits}
groups={name:{i['group'] for i in splits[name]['images']} for name in splits}
paths={name:{i['file_name'] for i in splits[name]['images']} for name in splits}
suffixes={name:{parse_file_name(i['orig_file_name'])[1] for i in splits[name]['images']} for name in main_names}
multisets={}
for name in main_names:
    cats={i['id']:[] for i in splits[name]['images']}
    for a in splits[name]['annotations']:cats[a['image_id']].append(a['category_id'])
    multisets[name]={sku_multiset(c) for c in cats.values() if c}
for k,name in enumerate(main_names):
    for other in main_names[k+1:]:
        assert not ids[name]&ids[other] and not groups[name]&groups[other] and not paths[name]&paths[other]
        assert not suffixes[name]&suffixes[other] and not multisets[name]&multisets[other]
assert set.union(*(ids[n] for n in main_names))=={i['id'] for i in merged['images']}
assert sum(len(splits[n]['annotations']) for n in main_names)==len(merged['annotations'])
assert ids['train_pilot']<=ids['train'] and groups['train_pilot']<=groups['train']
assert all(i['source']=='test2019' for n in ['val','test'] for i in splits[n]['images'])
assert {i['id'] for i in merged['images'] if i['source']=='val2019'}<=ids['train']
for name,minimum in [('train',1),('val',PARAMS['min_val_objects']),('test',PARAMS['min_test_objects'])]:
    counts=Counter(a['category_id'] for a in splits[name]['annotations'])
    assert set(counts)==set(range(1,201)) and min(counts.values())>=minimum,(name,min(counts.values()))
with ThreadPoolExecutor(max_workers=8) as pool:
    hashes=dict(zip((i['file_name'] for i in merged['images']),pool.map(lambda i:sha256(RAW/i['file_name']),merged['images']),strict=True))
hashsets={n:{hashes[p] for p in paths[n]} for n in main_names}
for k,name in enumerate(main_names):
    for other in main_names[k+1:]:assert not hashsets[name]&hashsets[other],(name,other,'duplicate image bytes')
background_names=[p.name for p in (RAW/'backgrounds').glob('*.jpg') if p.name!='preview.jpg']
assert len(background_names)==208
assert all('val2019/'+name in paths['train'] for name in background_names)
assert all('val2019/'+name not in paths['val']|paths['test'] for name in background_names)
save_json(ARTIFACT/'real_image_sha256.json',hashes)
checks['partition_groups_suffix_multiset']='PASS'
checks['all_200_sku_coverage']='PASS'
checks['no_cross_split_exact_hash_duplicate']='PASS'
checks['background_sources_train_only']='PASS'
print('PASS: full partition; group/suffix/SKU/hash disjoint; SKU coverage; all background source images stay in train')
print(json.dumps(split_cfg['counts'],indent=2))


PASS: full partition; group/suffix/SKU/hash disjoint; SKU coverage; all background source images stay in train
{
  "train": {
    "groups": 2504,
    "images": 22494,
    "objects": 276091,
    "images_per_level": {
      "easy": 7502,
      "medium": 7527,
      "hard": 7465
    },
    "min_objects_per_sku": 718
  },
  "val": {
    "groups": 167,
    "images": 1503,
    "objects": 18378,
    "images_per_level": {
      "easy": 490,
      "medium": 504,
      "hard": 509
    },
    "min_objects_per_sku": 18
  },
  "test": {
    "groups": 663,
    "images": 6003,
    "objects": 73466,
    "images_per_level": {
      "easy": 2008,
      "medium": 1969,
      "hard": 2026
    },
    "min_objects_per_sku": 124
  },
  "train_pilot": {
    "groups": 333,
    "images": 3002,
    "objects": 36987,
    "images_per_level": {
      "easy": 982,
      "medium": 1024,
      "hard": 996
    },
    "min_objects_per_sku": 87
  }
}


## 5. Sinh COCO cho từng task — chỉ train ảnh thật

Dùng make_task_json sẵn có. Train chỉ có label của đợt hiện tại, val_full/test_full giữ mọi label 0..199. File gt_full phục vụ audit, tuyệt đối không dùng làm input training. Capped tối đa 6.000 ảnh/task là tùy chọn pilot, không chọn sample bằng test performance.


In [6]:
TASK_DIR=OUT/'tasks/real'
make_task_json(['--task-config',str(OUT/'task_config.json'),'--train-source',f"real={OUT/'splits/train.json'}",
                '--val',str(OUT/'splits/val.json'),'--test',str(OUT/'splits/test.json'),
                '--out-dir',str(TASK_DIR),'--cap',str(PARAMS['cap']),'--seed',str(config.seed)])


train_task_1.json                images  21752  objects  138280
train_task_1_capped.json         images   6000  objects   38059
train_task_1_gt_full.json        images  21752  objects  270321
val_task_1.json                  images   1464  objects    9161
train_task_2.json                images  12534  objects   34197
train_task_2_capped.json         images   6000  objects   16365
train_task_2_gt_full.json        images  12534  objects  168030
val_task_2.json                  images    798  objects    2262
train_task_3.json                images  12129  objects   32320
train_task_3_capped.json         images   6000  objects   15963
train_task_3_gt_full.json        images  12129  objects  161088
val_task_3.json                  images    862  objects    2164
train_task_4.json                images  11850  objects   32859
train_task_4_capped.json         images   6000  objects   16533
train_task_4_gt_full.json        images  11850  objects  157141
val_task_4.json                  images 

## 6. Nghiệm thu nhãn/geometry/checksum và tái lập split

Kiểm tra từng file mới với annotation merged: mapping RPC↔model đúng, geometry không đổi, mọi task có đủ lớp hiện tại, không có label dự phòng trong train. Rebuild split với cùng input/seed phải có checksum giống nhau; test lock được ghi chỉ sau các checks này.


In [7]:
merged_by_ann={a['id']:a for a in merged['annotations']}
label_to_rpc=config.label_to_rpc();task_stats={}
for task in config.data_tasks:
    current=set(task.labels)
    stats={}
    for suffix in ['', '_capped', '_gt_full']:
        name=f'train_task_{task.task_id}{suffix}.json';d=load_json(TASK_DIR/name)
        assert {i['id'] for i in d['images']}<=ids['train']
        assert {i['train_source'] for i in d['images']}=={'real'}
        labels={a['category_id'] for a in d['annotations']}
        assert labels<=set(range(200))
        if suffix!='_gt_full':assert labels<=current
        if suffix=='':assert labels==current
        if suffix=='_capped':assert len(d['images'])<=PARAMS['cap']
        for a in d['annotations']:
            old=merged_by_ann[a['id']]
            assert label_to_rpc[a['category_id']]==old['category_id']
            assert a['bbox']==old['bbox'] and a['area']==old['area']
        stats[suffix or 'full']={'images':len(d['images']),'objects':len(d['annotations'])}
        del d
    d=load_json(TASK_DIR/f'val_task_{task.task_id}.json')
    assert {i['id'] for i in d['images']}<=ids['val']
    assert {a['category_id'] for a in d['annotations']}==current
    task_stats[str(task.task_id)]=stats
for name in ['val','test']:
    d=load_json(TASK_DIR/f'{name}_full.json')
    assert {i['id'] for i in d['images']}==ids[name]
    assert len(d['annotations'])==len(splits[name]['annotations'])
    assert {a['category_id'] for a in d['annotations']}==set(range(200))
    del d
manifest=load_json(TASK_DIR/'manifest.json')
assert manifest['tasks']==[1,2,3,4,5] and set(manifest['train_sources'])=={'real'}
for name,record in manifest['files'].items():assert record['md5']==md5_file(TASK_DIR/name)
REPEAT=ARTIFACT/'repeat_split'
make_split(['--ann',str(ANN),'--out-dir',str(REPEAT/'splits'),'--config-dir',str(REPEAT/'config'),
            *[a for k,v in PARAMS.items() if k!='cap' for a in ('--'+k.replace('_','-'),str(v))]])
for name in splits:assert md5_file(OUT/f'splits/{name}.json')==md5_file(REPEAT/f'splits/{name}.json')
checks['task_mapping_current_labels_native_geometry']='PASS'
checks['task_file_checksums']='PASS'
checks['split_reproducible_same_seed']='PASS'
print('PASS: label/geometry checks, checksum and byte-identical repeated split')
print(json.dumps(task_stats,indent=2))


train        groups   2504  images  22494  objects  276091  per level {'easy': 7502, 'medium': 7527, 'hard': 7465}
val          groups    167  images   1503  objects   18378  per level {'easy': 490, 'medium': 504, 'hard': 509}
test         groups    663  images   6003  objects   73466  per level {'easy': 2008, 'medium': 1969, 'hard': 2026}
train_pilot  groups    333  images   3002  objects   36987  per level {'easy': 982, 'medium': 1024, 'hard': 996}
seeds tried: [0]; accepted seed 0; 3323 groups span levels; wrote /home/sakana/Code/AutoCheckout-CL/runs/data_preprocessing/split/6b918406-79ea-4531-b3f0-76af0d6d265b/repeat_split/config/rpc_checkout_seed0.json
PASS: label/geometry checks, checksum and byte-identical repeated split
{
  "1": {
    "full": {
      "images": 21752,
      "objects": 138280
    },
    "_capped": {
      "images": 6000,
      "objects": 38059
    },
    "_gt_full": {
      "images": 21752,
      "objects": 270321
    }
  },
  "2": {
    "full": {
      "images":

## 7. Khóa release và kết thúc

Trạng thái đạt chỉ áp dụng **split/task annotations dữ liệu thật**, không phải nghiệm thu synth, processor 640/800 hoặc training. Không dùng số liệu thí nghiệm cũ; không đưa archive/raw ảnh vào git. Config lock có checksum, policy và danh sách đánh giá để tái lập trên Vast.ai.


In [8]:
assert checks and set(checks.values())=={'PASS'}
assert {name:sha256(RAW/name) for name in source_hashes}==source_hashes
assert sha256(CFG)==config_hash
run.update({'finished_at':now(),'status':'ACCEPTED_REAL_SPLIT_AND_TASKS'})
run['duration_seconds']=(datetime.fromisoformat(run['finished_at'])-datetime.fromisoformat(run['started_at'])).total_seconds()
release={'name':OUT.name,'input':release_input,'image_root':'data/archive','coordinates':'native_pixels',
         'protocol':[100,25,25,25,25],'seen_classes':[100,125,150,175,200],
         'num_slots':config.num_slots,'num_classes':config.num_classes,'split_seed':split_cfg['seed'],
         'counts':split_cfg['counts'],'task_counts':task_stats,'checks':checks,
         'split_md5':split_cfg['md5'],'task_manifest_sha256':sha256(TASK_DIR/'manifest.json'),
         'test_full_md5':md5_file(TASK_DIR/'test_full.json'),'synthetic_included':False,
         'limitations':['synth label/provenance gate pending','640/800 processor gate pending','no model training'],
         'run':run}
save_json(OUT/'release_manifest.json',release,indent=2)
LOCK=REPO/'configs/data'/f'{OUT.name}.json'
lock={k:v for k,v in release.items() if k not in ['run']}
lock['split_membership']=split_cfg['images']
lock['output_root']='data/processed/'+OUT.name
save_json(LOCK,lock,indent=1)
save_json(ARTIFACT/'summary.json',release,indent=2)
print(json.dumps(release,ensure_ascii=False,indent=2))
print('Config lock:',str(LOCK.relative_to(REPO)))


{
  "name": "rpc_100-4x25_seed0_native_v1",
  "input": {
    "source_sha256": {
      "instances_val2019.json": "25afdfed91bc09bff595399e0876a5707708a7061be3fa4121d13385abd1bde7",
      "instances_test2019.json": "2a1cb518b202c7e13a74b4ca742aad76f6246cba788288bac6423c7d4a97ba58"
    },
    "task_config_sha256": "be12b57f4c28b17597794ad526012a5ba3a72b98dd48739e2c31606f7240cf9a",
    "params": {
      "seed": 0,
      "max_tries": 20,
      "test_per_level": 2000,
      "val_per_level": 500,
      "pilot": 3000,
      "min_test_objects": 50,
      "min_val_objects": 15,
      "stratify": "none",
      "cap": 6000
    },
    "train_source": "real",
    "coordinates": "native_pixels"
  },
  "image_root": "data/archive",
  "coordinates": "native_pixels",
  "protocol": [
    100,
    25,
    25,
    25,
    25
  ],
  "seen_classes": [
    100,
    125,
    150,
    175,
    200
  ],
  "num_slots": 224,
  "num_classes": 225,
  "split_seed": 0,
  "counts": {
    "train": {
      "groups": 2504